## Init

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, sum
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler, PCA
from pyspark.sql.types import ArrayType, DoubleType

spark = SparkSession.builder.appName("SpotifyClustering").getOrCreate()

df_songs = spark.read.csv("hdfs://hadoop-namenode-1:8020/data/data.csv", header=True, inferSchema=True).dropDuplicates()

print("Spark session created and main dataset loaded successfully.")

df_songs.printSchema()
df_songs.show(5)

Spark session created and main dataset loaded successfully.
root
 |-- valence: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- explicit: string (nullable = true)
 |-- id: string (nullable = true)
 |-- instrumentalness: string (nullable = true)
 |-- key: string (nullable = true)
 |-- liveness: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- tempo: string (nullable = true)

+-------+----+------------+--------------------+------------+-----------+------+--------+--------------------+----------------+---+--------+--------+----+--------------------+

## Data cleaning & Feature preparation

In [2]:
feature_cols = [
    'acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness',
    'loudness', 'speechiness', 'tempo', 'valence', 'year'
]

for c in feature_cols:
    df_songs = df_songs.withColumn(c, col(c).cast("double"))

imputer = Imputer(inputCols=feature_cols, outputCols=feature_cols).setStrategy("mean")
df_imputed = imputer.fit(df_songs).transform(df_songs)

print("Data cleaning and imputation complete.")

df_imputed.show(5)

Data cleaning and imputation complete.
+-------+------+------------+--------------------+------------+-----------+------+--------+--------------------+----------------+---+--------+--------+----+--------------------+----------+------------+-----------+-------+
|valence|  year|acousticness|             artists|danceability|duration_ms|energy|explicit|                  id|instrumentalness|key|liveness|loudness|mode|                name|popularity|release_date|speechiness|  tempo|
+-------+------+------------+--------------------+------------+-----------+------+--------+--------------------+----------------+---+--------+--------+----+--------------------+----------+------------+-----------+-------+
|  0.965|1923.0|       0.907|['Louis Armstrong...|       0.573|     177307| 0.246|       0|2GnYkuVM8jQh4sFGG...|           0.867|  5|  0.0752| -13.553|   1|         Weary Blues|         4|        1923|      0.224|205.419|
|  0.327|1924.0|       0.996|['Francisco Canaro']|       0.424|     20405

In [3]:
print("Verifying null values after imputation...")

null_counts = df_imputed.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in feature_cols
])

null_counts.show()

Verifying null values after imputation...
+------------+------------+------+----------------+--------+--------+-----------+-----+-------+----+
|acousticness|danceability|energy|instrumentalness|liveness|loudness|speechiness|tempo|valence|year|
+------------+------------+------+----------------+--------+--------+-----------+-----+-------+----+
|           0|           0|     0|               0|       0|       0|          0|    0|      0|   0|
+------------+------------+------+----------------+--------+--------+-----------+-----+-------+----+



## Vector Assembly & Normalization

In [4]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="unscaled_features")
df_assembled = assembler.transform(df_imputed)

scaler = StandardScaler(inputCol="unscaled_features", outputCol="scaled_features", withStd=True, withMean=True)
df_scaled = scaler.fit(df_assembled).transform(df_assembled)

df_scaled.select("unscaled_features", "scaled_features").show(5, truncate=False)

print("Feature vector assembled and normalized.")

+-------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|unscaled_features                                                  |scaled_features                                                                                                                                                                                                 |
+-------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[0.907,0.573,0.246,0.867,0.0752,-13.553,0.224,205.419,0.965,1923.0]|[1.0766875860799618,0.20129458073126635,-0.03948991287920527,-0.011373377187380023,-0.00356703

## PCA

In [5]:
import numpy as np

pca = PCA(k=10, inputCol="scaled_features", outputCol="pca_features")
model_pca = pca.fit(df_scaled)

print("PCA complete.")

explained_variance = model_pca.explainedVariance.toArray()
cumulative_variance = np.cumsum(explained_variance)

print("\nExplained Variance Analysis:")
for i, (ev, cv) in enumerate(zip(explained_variance, cumulative_variance), 1):
    print(f"  Principal Component {i}: {ev*100:6.2f}% (Cumulative: {cv*100:6.2f}%)")

PCA complete.

Explained Variance Analysis:
  Principal Component 1:  24.10% (Cumulative:  24.10%)
  Principal Component 2:  13.31% (Cumulative:  37.41%)
  Principal Component 3:  12.60% (Cumulative:  50.01%)
  Principal Component 4:  10.01% (Cumulative:  60.02%)
  Principal Component 5:  10.00% (Cumulative:  70.02%)
  Principal Component 6:   9.63% (Cumulative:  79.65%)
  Principal Component 7:   7.79% (Cumulative:  87.44%)
  Principal Component 8:   5.44% (Cumulative:  92.88%)
  Principal Component 9:   3.92% (Cumulative:  96.80%)
  Principal Component 10:   3.20% (Cumulative: 100.00%)


In [6]:
pca = PCA(k=8, inputCol="scaled_features", outputCol="pca_features")
df_pca = pca.fit(df_scaled).transform(df_scaled)
print("PCA complete.")

PCA complete.


## Export

In [7]:
def vector_to_list(vector):
    return vector.toArray().tolist()

vector_to_list_udf = udf(vector_to_list, ArrayType(DoubleType()))

df_with_list = df_pca.withColumn("pca_features_list", vector_to_list_udf("pca_features"))

df_to_export = df_with_list.select(
    "name", "artists", "popularity", "year", "pca_features_list"
)

pandas_df = df_to_export.toPandas()
pandas_df.rename(columns={'pca_features_list': 'pca_features'}, inplace=True)
pandas_df.to_parquet("data_for_clustering.parquet")

print("\nFile successfully exported!")

pandas_df.head(10)


File successfully exported!


,name,artists,popularity,year,pca_features
0,Weary Blues,['Louis Armstrong & His Hot Seven'],4,1923.0,"[-0.9500305971402666, -2.4371640540862045, 0.1..."
1,Sobin Blue - Remasterizado,['Francisco Canaro'],0,1924.0,"[-2.573148077462789, -0.6122167247591741, 0.15..."
2,Ojerosa - Remasterizado,['Ignacio Corsini'],0,1925.0,"[-2.6386979402087483, -2.0897905198270084, 0.0..."
3,No Folling - Instrumental (Remasterizado),['Francisco Canaro'],0,1927.0,"[-1.140503933490412, -3.214696881959803, -0.50..."
4,En la Cortada - Remasterizado,['Ignacio Corsini'],0,1927.0,"[-2.8109864365197907, -1.6004475993935068, 0.0..."
5,After the Ball,"['Barbara Cook', 'The Merrill Staton Choir']",3,1928.0,"[-2.489839346303047, 0.21765050802054908, 0.10..."
6,Blue Is the Night,"['Jack Teagarden', 'Ben Pollack & His Orchestra']",8,1930.0,"[-2.154136501557534, -0.6575950430270743, -0.0..."
7,Avalon - Take 2,['Benny Goodman Quartet'],7,1935.0,"[-0.43817894872164503, -2.9487019929816385, -0..."
8,"Das ist bei uns nicht möglich, Kapitel 206","['Sinclair Lewis', 'Frank Arnold']",0,1935.0,"[-1.6433318241094577, -1.3332619896041047, -0...."
9,"Wie man Freunde gewinnt - Die Kunst, beliebt u...","['Dale Carnegie', 'Till Hagen', 'Stefan Kamins...",17,1936.0,"[-0.6533509266455385, -1.2694624313956808, -0...."
